# Station workplace testbench

Interactive testbench for the `workplace_rfid_scanner` lifecycle node, focused on the `~/wait_routing` action. Everything runs simulated / in local mode: Gazebo (conveyor + RFID scanner), MoveIt 2 (`move_group`), the `conveyor`, `simatic_rf240r_ascii`, `production_manager` and `object_model_storage` nodes.

Prerequisites:
1. Build the workspace (`colcon build`) and use a Jupyter kernel from `/opt/venv` with ROS 2 Jazzy and the workspace overlay sourced.
2. Start the testbench in a terminal; it copies `assets/` to `/tmp/station` and launches everything (Gazebo GUI and RViz included):
   ```bash
   src/station_workplace/tests/testbench/setup.sh
   ```
3. Make sure the kernel uses the same `ROS_DOMAIN_ID` as the launch terminal (`setup.sh` defaults to `42`).

All lifecycle nodes start **Unconfigured**; use the cells below to drive them. `~/wait_routing`, `~/spawn_product` and `~/remove_product` exist only while the workplace node is **Active**.

Scenario: bring up the dependencies and the workplace node → set the operation → spawn the product → send an `injection` goal and move the carrier in → send an `ejection` goal and move the carrier out → remove the product.

The carrier is moved manually in the Gazebo GUI: ⋮ menu → **Joint Position Controller**, select the model `station`, set `base_carrier_joint` to `-0.6` (in place) or `0.0` (out).

## Setup

In [ ]:
import threading
import time

import rclpy
from action_msgs.msg import GoalStatus
from lifecycle_msgs.msg import Transition
from lifecycle_msgs.srv import ChangeState, GetAvailableTransitions, GetState
from rclpy.action import ActionClient
from rclpy.executors import SingleThreadedExecutor
from rclpy.qos import DurabilityPolicy, QoSProfile, ReliabilityPolicy

from production_manager.srv import SetOperation
from station_workplace.action import WaitRouting
from station_workplace.msg import WorkplaceState
from station_workplace.srv import RemoveProduct, SpawnProduct

TARGET_NODE = '/workplace_rfid_scanner'
DEPENDENCY_NODES = ['/conveyor', '/simatic_rf240r_ascii', '/production_manager', '/object_model_storage']

# Must match testbench.launch.py (station_id) and assets/mock_assembly_file.json
STATION_ID = 'c330289c-0db3-4b5b-9fe0-3d9b63bedb72'
OPERATION_ID = 'dab06081-0e24-4583-9f33-06d3c26b69f7'
PRODUCT = {'product_number': '12345', 'product_revision': '1.0', 'serial_number': 'SN-0001'}
# SDF product model (assets/mock_product) and its pose relative to the carrier's carrier_link
# [x, y, z, roll, pitch, yaw]: box bottom centre on the middle of the pallet top
ASSEMBLY_MODEL_ID = 'fd71afb0-d3e9-428c-a7e6-b2613b877088'
SPAWN_POSE = [-0.35, 0.25, 0.02, 0.0, 0.0, 0.0]

if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node('workplace_testbench')
executor = SingleThreadedExecutor()
executor.add_node(node)
spin_thread = threading.Thread(target=executor.spin, daemon=True)
spin_thread.start()

In [ ]:
def call(service_type, service_name, request, timeout=10.0):
    client = node.create_client(service_type, service_name)
    try:
        if not client.wait_for_service(timeout_sec=timeout):
            raise TimeoutError(f'Service {service_name} was not available within {timeout}s')
        future = client.call_async(request)
        deadline = time.monotonic() + timeout
        while not future.done():
            if time.monotonic() > deadline:
                raise TimeoutError(f'Service call {service_name} did not complete within {timeout}s')
            time.sleep(0.01)
        return future.result()
    finally:
        node.destroy_client(client)

def get_state(target=TARGET_NODE):
    return call(GetState, f'{target}/get_state', GetState.Request()).current_state.label

def available_transitions(target=TARGET_NODE):
    response = call(GetAvailableTransitions, f'{target}/get_available_transitions', GetAvailableTransitions.Request())
    return [t.transition.label for t in response.available_transitions]

def transition(transition_id, target=TARGET_NODE):
    request = ChangeState.Request()
    request.transition.id = transition_id
    success = call(ChangeState, f'{target}/change_state', request).success
    print(f'{target}: success={success}, state={get_state(target)}')

def bring_up(target):
    """Configure (if needed) and activate a lifecycle node; an active node is left as is."""
    if get_state(target) == 'unconfigured':
        transition(Transition.TRANSITION_CONFIGURE, target)
    if get_state(target) == 'inactive':
        transition(Transition.TRANSITION_ACTIVATE, target)
    print(f'{target}: {get_state(target)}')

## Monitor

Subscribes to the node's latched `~/workplace` topic; changes are printed as they arrive. Inspect `latest` at any time.

In [ ]:
latest = {'workplace': None}

def _on_workplace(msg):
    latest['workplace'] = msg
    print(f'[workplace] in_place={msg.in_place} product_data={list(msg.product_data)}')

latched_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE, durability=DurabilityPolicy.TRANSIENT_LOCAL)
workplace_sub = node.create_subscription(WorkplaceState, f'{TARGET_NODE}/workplace', _on_workplace, latched_qos)

In [ ]:
latest

## Dependencies: bring up

Configures and activates the conveyor, RFID scanner, production manager and object model storage. Safe to re-run: active nodes are skipped.

In [ ]:
for dependency in DEPENDENCY_NODES:
    bring_up(dependency)

## Lifecycle: bring up

In [ ]:
print(get_state(), available_transitions())

In [ ]:
transition(Transition.TRANSITION_CONFIGURE)

In [ ]:
transition(Transition.TRANSITION_ACTIVATE)

## Production manager: set operation

On injection the workplace node calls `get_production_data`, which only answers for the operation and serial number set here (and the production manager's `station_id`).

In [ ]:
request = SetOperation.Request(operation_id=OPERATION_ID, serial_number=PRODUCT['serial_number'], station_id=STATION_ID)
response = call(SetOperation, '/production_manager/set_operation', request)
print(f'success={response.success} error_message={response.error_message!r}')

## Simulation: spawn product (node must be Active)

Spawns the carrier (pallet, with `PRODUCT` written to its RFID tag) at the current pose of the conveyor `carrier` link and the product on top of it; both are welded on spawn. Spawn while the carrier is **out** (`base_carrier_joint` = `0.0`). Only one product can be spawned at a time; remove it first.

In [ ]:
request = SpawnProduct.Request(**PRODUCT, assembly_model_id=ASSEMBLY_MODEL_ID, pose=SPAWN_POSE)
response = call(SpawnProduct, f'{TARGET_NODE}/spawn_product', request, timeout=30.0)
print(f'success={response.success} error_message={response.error_message!r}')

## wait_routing action (node must be Active)

`send_wait_routing()` sends a goal and returns its goal handle; the result is printed when it arrives (status, `routing_time_s`, `success`, `error_message`, `product_data`). Only one goal runs at a time; a second goal or an invalid `routing_operation` is rejected.

With the simulated conveyor, `inject_product` / `eject_product` are no-ops: after sending the goal, move the carrier in Gazebo (Joint Position Controller, model `station`, `base_carrier_joint`: `-0.6` for `injection`, `0.0` for `ejection`) to trigger the injection / ejection the goal waits for. On injection the carrier and product appear in RViz's PlanningScene, attached to `carrier`, together with the `estimated_product_origin` TF frame.

In [ ]:
STATUS = {GoalStatus.STATUS_SUCCEEDED: 'SUCCEEDED', GoalStatus.STATUS_ABORTED: 'ABORTED',
          GoalStatus.STATUS_CANCELED: 'CANCELED'}
wait_routing_client = ActionClient(node, WaitRouting, f'{TARGET_NODE}/wait_routing')

def _print_result(future):
    response = future.result()
    result = response.result
    print(f'[wait_routing] status={STATUS.get(response.status, response.status)} '
          f'routing_time_s={result.routing_time_s} success={result.success} '
          f'error_message={result.error_message!r} product_data={list(result.product_data)}')

def send_wait_routing(routing_operation, timeout=5.0):
    if not wait_routing_client.wait_for_server(timeout_sec=timeout):
        raise TimeoutError(f'{TARGET_NODE}/wait_routing not available (is the node active?)')
    future = wait_routing_client.send_goal_async(WaitRouting.Goal(routing_operation=routing_operation))
    deadline = time.monotonic() + timeout
    while not future.done():
        if time.monotonic() > deadline:
            raise TimeoutError('Goal response not received')
        time.sleep(0.01)
    goal_handle = future.result()
    print(f'[wait_routing] {routing_operation}: accepted={goal_handle.accepted}')
    if goal_handle.accepted:
        goal_handle.get_result_async().add_done_callback(_print_result)
    return goal_handle

In [ ]:
goal = send_wait_routing(WaitRouting.Goal.ROUTING_OPERATION_INJECTION)

In [ ]:
goal = send_wait_routing(WaitRouting.Goal.ROUTING_OPERATION_EJECTION)

In [ ]:
# Cancel the last goal (only stops the wait; the carrier operation is not affected)
goal.cancel_goal_async()

In [ ]:
# Rejected: invalid routing_operation
send_wait_routing('teleport')

## Simulation: remove product

Detaches and removes the spawned product and carrier models (product first). Move the carrier out (and finish the ejection) first.

In [ ]:
response = call(RemoveProduct, f'{TARGET_NODE}/remove_product', RemoveProduct.Request(), timeout=30.0)
print(f'success={response.success} error_message={response.error_message!r}')

## Lifecycle: bring down

Deactivating while a goal is running aborts it (`wait_routing aborted: node deactivated`).

In [ ]:
transition(Transition.TRANSITION_DEACTIVATE)

In [ ]:
transition(Transition.TRANSITION_CLEANUP)

In [ ]:
# Final: the node cannot be brought back without restarting it
# transition(Transition.TRANSITION_UNCONFIGURED_SHUTDOWN)

In [ ]:
# Optional: bring the dependencies down as well
for dependency in reversed(DEPENDENCY_NODES):
    transition(Transition.TRANSITION_DEACTIVATE, dependency)
    transition(Transition.TRANSITION_CLEANUP, dependency)

## Teardown

In [ ]:
wait_routing_client.destroy()
node.destroy_subscription(workplace_sub)
executor.shutdown()
spin_thread.join(timeout=5.0)
node.destroy_node()
rclpy.shutdown()